# Model Diagnostics & OLS Assumption Tests

Comprehensive diagnostic tests for the final 6-variable oil price forecasting model.

Tests performed in order:
1. Functional Form (Ramsey RESET)
2. Autocorrelation (Breusch-Godfrey Test)
3. Heteroskedasticity
4. Normality
5. Multicollinearity
6. Structural Breaks & Outliers

## Setup & Load Model

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.stats.diagnostic import acorr_breusch_godfrey as bg_test
from scipy.stats import jarque_bera, shapiro
import warnings
warnings.filterwarnings('ignore')

# Load data and fit model
IN_SAMPLE_FILE = Path("Oil_InSample.csv")

df_insample = pd.read_csv(IN_SAMPLE_FILE, index_col=0, parse_dates=True)

# Final variables
final_variables = [
    'dow_jones_us_oilandgas_index_ret',
    'exxonmobil_ret',
    'cl1_ret_l2',
    'weekly_commercial_short_diff',
    'weekly_commercial_long_diff',
    'weekly_change_in_crude_stock_diff'
]

y = df_insample['cl1_ret']
X = df_insample[final_variables]
X_const = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X_const).fit()
residuals = model.resid
predictions = model.predict(X_const)

print("="*90)
print("MODEL DIAGNOSTICS - OLS ASSUMPTION TESTS")
print("="*90)
print(f"\nSample size: {len(y)}")
print(f"Number of variables: {len(final_variables)}")
print(f"Observations: {len(y)}\n")

MODEL DIAGNOSTICS - OLS ASSUMPTION TESTS

Sample size: 1195
Number of variables: 6
Observations: 1195



# STEP 1: FUNCTIONAL FORM

Tests whether the linear specification is appropriate or if non-linear terms are needed.

In [11]:
print("\n" + "="*90)
print("STEP 1: FUNCTIONAL FORM TEST")
print("="*90)

print("\nPURPOSE: Verify that the linear functional form is appropriate")
print("QUESTION: Is the model y = β₀ + β₁X + ε the right specification?\n")

# Test 1.1: Ramsey RESET Test
print("-" * 90)
print("TEST 1.1: RAMSEY RESET TEST (Specification Error Test)")
print("-" * 90)
print("Tests if the linear functional form is correct by examining powers of fitted values\n")

# Perform RESET test
reset_test = linear_reset(model)

print("Test Statistic (F):" + reset_test[0])
print(f"P-value: {reset_test[1]:.6f}")
print(f"Degrees of Freedom: ({reset_test[2]}, {reset_test[3]})")

print(f"\nH0: Linear model is correct (no omitted polynomial terms)")
print(f"H1: Non-linear terms (powers of fitted values) are needed")

if reset_test[1] > 0.05:
    print(f"\nResult: ✓ PASS - Fail to reject H0 (p={reset_test[1]:.4f})")
    print(f"Conclusion: Linear functional form is appropriate for this data")
    functional_form_pass = True
else:
    print(f"\nResult: ✗ FAIL - Reject H0 (p={reset_test[1]:.4f})")
    print(f"Conclusion: Model may need non-linear terms or polynomial terms")
    print(f"Remedies: Consider adding polynomial terms (X²) or interaction effects")
    functional_form_pass = False

print("\n" + "="*90)
print(f"FUNCTIONAL FORM SUMMARY: {'✓ LINEAR FORM APPROPRIATE' if functional_form_pass else '✗ ISSUES DETECTED'}")
print("="*90)


STEP 1: FUNCTIONAL FORM TEST

PURPOSE: Verify that the linear functional form is appropriate
QUESTION: Is the model y = β₀ + β₁X + ε the right specification?

------------------------------------------------------------------------------------------
TEST 1.1: RAMSEY RESET TEST (Specification Error Test)
------------------------------------------------------------------------------------------
Tests if the linear functional form is correct by examining powers of fitted values



TypeError: 'ContrastResults' object is not subscriptable

# STEP 2: AUTOCORRELATION

Tests whether residuals are independent over time.

In [4]:
print("\n" + "="*90)
print("STEP 2: AUTOCORRELATION TEST")
print("="*90)

print("\nPURPOSE: Verify that residuals are independent (not correlated over time)")
print("QUESTION: Do prediction errors in one period depend on errors in previous periods?\n")

# Test 2.1: Breusch-Godfrey Test
print("-" * 90)
print("TEST 2.1: BREUSCH-GODFREY TEST (Lagrange Multiplier Test for Autocorrelation)")
print("-" * 90)
print("Tests for autocorrelation of any order (more general than Durbin-Watson)\n")

# Perform BG test at different lags
lags_to_test = [1, 5, 10]
bg_results = {}

for lag in lags_to_test:
    bg_test = bgtest(residuals, nlags=lag)
    bg_results[lag] = {
        'test_stat': bg_test[0],
        'p_value': bg_test[1],
        'df': bg_test[2]
    }

print(f"{'Lag':<8} {'Test Stat (LM)':<20} {'P-Value':<20} {'Result':<15}")
print("-" * 60)

bg_pass = True
for lag in lags_to_test:
    result = "✓ PASS" if bg_results[lag]['p_value'] > 0.05 else "✗ FAIL"
    print(f"{lag:<8} {bg_results[lag]['test_stat']:<20.6f} {bg_results[lag]['p_value']:<20.6f} {result:<15}")
    if bg_results[lag]['p_value'] <= 0.05:
        bg_pass = False

print(f"\nH0: No autocorrelation of any order")
print(f"H1: Autocorrelation present")

if bg_pass:
    print(f"\nResult: ✓ PASS - No autocorrelation detected at any tested lag")
    print(f"Conclusion: Residuals appear independent")
    autocorr_pass = True
else:
    print(f"\nResult: ✗ FAIL - Autocorrelation detected")
    sig_lags = [lag for lag in lags_to_test if bg_results[lag]['p_value'] <= 0.05]
    print(f"Significant at lags: {sig_lags}")
    print(f"Remedies: Add lagged dependent variable, use HAC standard errors, or ARIMA model")
    autocorr_pass = False

print("\n" + "="*90)
print(f"AUTOCORRELATION SUMMARY: {'✓ NOT DETECTED' if autocorr_pass else '✗ DETECTED'}")
print("="*90)


STEP 2: AUTOCORRELATION TEST

PURPOSE: Verify that residuals are independent (not correlated over time)
QUESTION: Do prediction errors in one period depend on errors in previous periods?

------------------------------------------------------------------------------------------
TEST 2.1: BREUSCH-GODFREY TEST (Lagrange Multiplier Test for Autocorrelation)
------------------------------------------------------------------------------------------
Tests for autocorrelation of any order (more general than Durbin-Watson)



NameError: name 'bgtest' is not defined

# STEP 3: HETEROSKEDASTICITY

Tests whether error variance is constant across observations.

In [5]:
print("\n" + "="*90)
print("STEP 3: HETEROSKEDASTICITY TESTS")
print("="*90)

print("\nPURPOSE: Verify that error variance is constant (homoskedastic)")
print("QUESTION: Does the prediction error magnitude stay constant or vary?\n")

# Test 3.1: Breusch-Pagan Test
print("-" * 90)
print("TEST 3.1: BREUSCH-PAGAN TEST")
print("-" * 90)
print("Tests if variance changes with regressors\n")

bp_test = het_breuschpagan(residuals, X_const)

print(f"Test Statistic (LM): {bp_test[0]:.6f}")
print(f"P-value: {bp_test[1]:.6f}")
print(f"F-statistic: {bp_test[2]:.6f}")
print(f"F-stat p-value: {bp_test[3]:.6f}")

print(f"\nH0: Homoskedastic (constant variance)")
print(f"H1: Heteroskedastic (variance changes)")

if bp_test[1] > 0.05:
    print(f"\nResult: ✓ PASS - No heteroskedasticity (p={bp_test[1]:.4f})")
    print(f"Conclusion: Error variance appears constant")
    bp_pass = True
else:
    print(f"\nResult: ✗ FAIL - Heteroskedasticity detected (p={bp_test[1]:.4f})")
    print(f"Remedies: Use robust standard errors (HC3, HC4) or Weighted Least Squares")
    bp_pass = False

# Test 3.2: White's Test
print("\n" + "-" * 90)
print("TEST 3.2: WHITE'S TEST")
print("-" * 90)
print("General test for heteroskedasticity (includes squared terms and interactions)\n")

white_test = het_white(residuals, X_const)

print(f"Test Statistic (LM): {white_test[0]:.6f}")
print(f"P-value: {white_test[1]:.6f}")

print(f"\nH0: Homoskedastic (constant variance)")
print(f"H1: Heteroskedastic (variance changes)")

if white_test[1] > 0.05:
    print(f"\nResult: ✓ PASS - No heteroskedasticity detected (p={white_test[1]:.4f})")
    white_pass = True
else:
    print(f"\nResult: ✗ FAIL - Heteroskedasticity detected (p={white_test[1]:.4f})")
    white_pass = False

heterosk_pass = bp_pass and white_pass

print("\n" + "="*90)
print(f"HETEROSKEDASTICITY SUMMARY: {'✓ NOT PRESENT' if heterosk_pass else '✗ PRESENT'}")
print("="*90)


STEP 3: HETEROSKEDASTICITY TESTS

PURPOSE: Verify that error variance is constant (homoskedastic)
QUESTION: Does the prediction error magnitude stay constant or vary?

------------------------------------------------------------------------------------------
TEST 3.1: BREUSCH-PAGAN TEST
------------------------------------------------------------------------------------------
Tests if variance changes with regressors

Test Statistic (LM): 70.275785
P-value: 0.000000
F-statistic: 12.371571
F-stat p-value: 0.000000

H0: Homoskedastic (constant variance)
H1: Heteroskedastic (variance changes)

Result: ✗ FAIL - Heteroskedasticity detected (p=0.0000)
Remedies: Use robust standard errors (HC3, HC4) or Weighted Least Squares

------------------------------------------------------------------------------------------
TEST 3.2: WHITE'S TEST
------------------------------------------------------------------------------------------
General test for heteroskedasticity (includes squared terms and i

AssertionError: 

# STEP 4: NORMALITY

Tests whether residuals are normally distributed.

In [6]:
print("\n" + "="*90)
print("STEP 4: NORMALITY TESTS")
print("="*90)

print("\nPURPOSE: Verify that residuals follow a normal distribution")
print("QUESTION: Do errors follow a bell curve or are they skewed/fat-tailed?\n")

# Test 4.1: Jarque-Bera Test
print("-" * 90)
print("TEST 4.1: JARQUE-BERA TEST")
print("-" * 90)
print("Tests skewness and kurtosis of residuals\n")

jb_stat, jb_pval = jarque_bera(residuals)

print(f"Test Statistic: {jb_stat:.6f}")
print(f"P-value: {jb_pval:.6f}")

print(f"\nResidual Properties:")
print(f"  Skewness: {residuals.skew():.6f} {'(acceptable < 0.5)' if abs(residuals.skew()) < 0.5 else '(concerning > 0.5)'}")
print(f"  Kurtosis: {residuals.kurtosis():.6f} {'(acceptable < 1)' if abs(residuals.kurtosis()) < 1 else '(concerning > 1)'}")

print(f"\nH0: Residuals are normally distributed")
print(f"H1: Residuals are not normal")

if jb_pval > 0.05:
    print(f"\nResult: ✓ PASS - Residuals appear normally distributed (p={jb_pval:.4f})")
    jb_pass = True
else:
    print(f"\nResult: ✗ FAIL - Residuals deviate from normality (p={jb_pval:.4f})")
    if abs(residuals.skew()) > 0.5:
        print(f"  Issue: Significant skewness ({residuals.skew():.4f})")
    if abs(residuals.kurtosis()) > 1:
        print(f"  Issue: Significant kurtosis - fat tails ({residuals.kurtosis():.4f})")
    jb_pass = False

# Test 4.2: Shapiro-Wilk Test
print("\n" + "-" * 90)
print("TEST 4.2: SHAPIRO-WILK TEST")
print("-" * 90)
print("Alternative normality test (recommended for smaller samples)\n")

sample_residuals = residuals.sample(min(5000, len(residuals)), random_state=42)
sw_stat, sw_pval = shapiro(sample_residuals)

print(f"Test Statistic: {sw_stat:.6f}")
print(f"P-value: {sw_pval:.6f}")
print(f"Sample Size: {len(sample_residuals)}")

print(f"\nH0: Residuals are normally distributed")
print(f"H1: Residuals are not normal")

if sw_pval > 0.05:
    print(f"\nResult: ✓ PASS - Residuals are normal (p={sw_pval:.4f})")
    sw_pass = True
else:
    print(f"\nResult: ✗ FAIL - Residuals deviate from normality (p={sw_pval:.4f})")
    sw_pass = False

normality_pass = jb_pass and sw_pass

print("\n" + "="*90)
print(f"NORMALITY SUMMARY: {'✓ SATISFIED' if normality_pass else '✗ VIOLATED'}")
print("="*90)


STEP 4: NORMALITY TESTS

PURPOSE: Verify that residuals follow a normal distribution
QUESTION: Do errors follow a bell curve or are they skewed/fat-tailed?

------------------------------------------------------------------------------------------
TEST 4.1: JARQUE-BERA TEST
------------------------------------------------------------------------------------------
Tests skewness and kurtosis of residuals

Test Statistic: 1064.531023
P-value: 0.000000

Residual Properties:
  Skewness: -0.670965 (concerning > 0.5)
  Kurtosis: 4.448929 (concerning > 1)

H0: Residuals are normally distributed
H1: Residuals are not normal

Result: ✗ FAIL - Residuals deviate from normality (p=0.0000)
  Issue: Significant skewness (-0.6710)
  Issue: Significant kurtosis - fat tails (4.4489)

------------------------------------------------------------------------------------------
TEST 4.2: SHAPIRO-WILK TEST
------------------------------------------------------------------------------------------
Alternative

# STEP 5: MULTICOLLINEARITY

Tests whether independent variables are too highly correlated.

In [7]:
print("\n" + "="*90)
print("STEP 5: MULTICOLLINEARITY TESTS")
print("="*90)

print("\nPURPOSE: Verify that explanatory variables are not too highly correlated")
print("QUESTION: Are the 6 variables measuring different things or are they redundant?\n")

# Test 5.1: Variance Inflation Factor
print("-" * 90)
print("TEST 5.1: VARIANCE INFLATION FACTOR (VIF)")
print("-" * 90)
print("Measures multicollinearity for each variable\n")

vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False)

print(f"{'Variable':<45} {'VIF':<15}")
print("-" * 60)
for idx, row in vif_data.iterrows():
    status = "✓" if row['VIF'] <= 5 else "✗" if row['VIF'] > 10 else "⚠"
    print(f"{status} {row['Variable']:<43} {row['VIF']:<15.2f}")

print(f"\nInterpretation Scale:")
print(f"  VIF = 1: No correlation with other variables")
print(f"  VIF 1-5: Acceptable correlation")
print(f"  VIF 5-10: Moderate concern")
print(f"  VIF > 10: Serious multicollinearity")

if all(vif_data['VIF'] <= 5):
    print(f"\nResult: ✓ PASS - No serious multicollinearity")
    vif_pass = True
else:
    print(f"\nResult: ✗ FAIL - Multicollinearity detected")
    vif_pass = False

# Test 5.2: Condition Number
print("\n" + "-" * 90)
print("TEST 5.2: CONDITION NUMBER")
print("-" * 90)
print("Overall measure of multicollinearity across all variables\n")

cond_num = model.condition_number

print(f"Condition Number: {cond_num:.2e}")

print(f"\nInterpretation Scale:")
print(f"  < 20: Low multicollinearity (EXCELLENT)")
print(f"  20-100: Moderate multicollinearity (ACCEPTABLE)")
print(f"  > 100: High multicollinearity (PROBLEMATIC)")

if cond_num < 20:
    print(f"\nResult: ✓ EXCELLENT - Low multicollinearity")
    cond_pass = True
elif cond_num < 100:
    print(f"\nResult: ✓ ACCEPTABLE - Moderate multicollinearity")
    cond_pass = True
else:
    print(f"\nResult: ✗ PROBLEMATIC - High multicollinearity")
    cond_pass = False

multicolli_pass = vif_pass and cond_pass

print("\n" + "="*90)
print(f"MULTICOLLINEARITY SUMMARY: {'✓ EXCELLENT/ACCEPTABLE' if multicolli_pass else '✗ PROBLEMATIC'}")
print("="*90)


STEP 5: MULTICOLLINEARITY TESTS

PURPOSE: Verify that explanatory variables are not too highly correlated
QUESTION: Are the 6 variables measuring different things or are they redundant?

------------------------------------------------------------------------------------------
TEST 5.1: VARIANCE INFLATION FACTOR (VIF)
------------------------------------------------------------------------------------------
Measures multicollinearity for each variable

Variable                                      VIF            
------------------------------------------------------------
✓ dow_jones_us_oilandgas_index_ret            2.71           
✓ exxonmobil_ret                              2.70           
✓ weekly_commercial_short_diff                1.73           
✓ weekly_commercial_long_diff                 1.72           
✓ cl1_ret_l2                                  1.04           
✓ weekly_change_in_crude_stock_diff           1.02           

Interpretation Scale:
  VIF = 1: No correlatio

# STEP 6: STRUCTURAL BREAKS & STABILITY

Tests for outliers, influential observations, and structural stability.

In [8]:
print("\n" + "="*90)
print("STEP 6: STRUCTURAL BREAKS & STABILITY")
print("="*90)

print("\nPURPOSE: Identify extreme observations and structural breaks")
print("QUESTION: Are there outliers or periods where the model breaks down?\n")

# Test 6.1: Cook's Distance
print("-" * 90)
print("TEST 6.1: COOK'S DISTANCE")
print("-" * 90)
print("Measures how much each observation influences the regression coefficients\n")

influence = OLSInfluence(model)
cooks_d = influence.cooks_distance[0]

threshold = 4 / len(y)
influential_idx = np.where(cooks_d > threshold)[0]

print(f"Threshold: {threshold:.6f}")
print(f"Influential observations (above threshold): {len(influential_idx)}")

if len(influential_idx) > 0:
    print(f"\nTop 10 Most Influential:")
    top_influential = np.argsort(cooks_d)[-10:][::-1]
    for i, idx in enumerate(top_influential, 1):
        date = df_insample.index[idx].date()
        print(f"  {i}. Index {idx} ({date}): Cook's D = {cooks_d[idx]:.6f}")
    print(f"\nResult: ⚠ WARNING - Influential observations detected")
    cooks_pass = False
else:
    print(f"\nResult: ✓ PASS - No highly influential observations")
    cooks_pass = True

# Test 6.2: Leverage (Hat Values)
print("\n" + "-" * 90)
print("TEST 6.2: LEVERAGE (HAT VALUES)")
print("-" * 90)
print("Identifies extreme values in X space that may distort results\n")

leverages = influence.hat_matrix_diag
avg_leverage = np.mean(leverages)
threshold_leverage = 2 * avg_leverage
high_leverage_idx = np.where(leverages > threshold_leverage)[0]

print(f"Average Leverage: {avg_leverage:.6f}")
print(f"Threshold (2x average): {threshold_leverage:.6f}")
print(f"High-leverage observations: {len(high_leverage_idx)}")

if len(high_leverage_idx) > 0:
    print(f"\nTop 10 High-Leverage:")
    top_leverage = np.argsort(leverages)[-10:][::-1]
    for i, idx in enumerate(top_leverage, 1):
        date = df_insample.index[idx].date()
        print(f"  {i}. Index {idx} ({date}): Leverage = {leverages[idx]:.6f}")
    print(f"\nResult: ⚠ WARNING - High-leverage points detected")
    leverage_pass = False
else:
    print(f"\nResult: ✓ PASS - No problematic high-leverage points")
    leverage_pass = True

# Test 6.3: Residual Statistics
print("\n" + "-" * 90)
print("TEST 6.3: RESIDUAL STATISTICS & OUTLIERS")
print("-" * 90)
print("Identifies extreme residuals (potential outliers)\n")

standardized_resid = (residuals - residuals.mean()) / residuals.std()
outliers = np.where(np.abs(standardized_resid) > 3)[0]

print(f"Residual Properties:")
print(f"  Mean: {residuals.mean():.8f}")
print(f"  Std Dev: {residuals.std():.6f}")
print(f"  Min: {residuals.min():.6f}")
print(f"  Max: {residuals.max():.6f}")

print(f"\nOutliers (|standardized residual| > 3): {len(outliers)}")

if len(outliers) > 0:
    print(f"\nOutlier Dates:")
    for i, idx in enumerate(outliers[:10], 1):
        date = df_insample.index[idx].date()
        print(f"  {i}. {date}: residual = {residuals.iloc[idx]:.6f}")
    if len(outliers) > 10:
        print(f"  ... and {len(outliers) - 10} more")
    print(f"\nResult: ⚠ WARNING - Outliers present")
    outliers_pass = False
else:
    print(f"\nResult: ✓ PASS - No extreme outliers")
    outliers_pass = True

structural_pass = cooks_pass and leverage_pass and outliers_pass

print("\n" + "="*90)
print(f"STRUCTURAL STABILITY SUMMARY: {'✓ STABLE' if structural_pass else '⚠ ISSUES FOUND'}")
print("="*90)


STEP 6: STRUCTURAL BREAKS & STABILITY

PURPOSE: Identify extreme observations and structural breaks
QUESTION: Are there outliers or periods where the model breaks down?

------------------------------------------------------------------------------------------
TEST 6.1: COOK'S DISTANCE
------------------------------------------------------------------------------------------
Measures how much each observation influences the regression coefficients

Threshold: 0.003347
Influential observations (above threshold): 69

Top 10 Most Influential:
  1. Index 723 (2008-12-19): Cook's D = 0.300365
  2. Index 721 (2008-12-05): Cook's D = 0.159544
  3. Index 725 (2009-01-02): Cook's D = 0.076243
  4. Index 713 (2008-10-10): Cook's D = 0.045211
  5. Index 869 (2011-10-07): Cook's D = 0.025212
  6. Index 423 (2003-03-21): Cook's D = 0.024341
  7. Index 728 (2009-01-23): Cook's D = 0.024187
  8. Index 176 (1998-06-26): Cook's D = 0.020260
  9. Index 719 (2008-11-21): Cook's D = 0.019935
  10. Index 

# FINAL DIAGNOSTIC SUMMARY

In [9]:
print("\n\n" + "="*90)
print("COMPREHENSIVE DIAGNOSTIC SUMMARY")
print("="*90)

summary_table = f"""
╔════════════════════════════════════════════════════════════════════════════════════════╗
║                            OLS ASSUMPTION TEST RESULTS                                 ║
╚════════════════════════════════════════════════════════════════════════════════════════╝

1. FUNCTIONAL FORM (Ramsey RESET)
   Test Statistic: {linear_reset(model)[0]:.6f}
   P-Value: {linear_reset(model)[1]:.6f}
   Status: {'✓ PASS - Linear form is appropriate' if linear_reset(model)[1] > 0.05 else '✗ FAIL'}

2. AUTOCORRELATION (Breusch-Godfrey Test)
   BG Test Results (selected lags):
     Lag 1: p = {bg_results[1]['p_value']:.6f} {'✓ PASS' if bg_results[1]['p_value'] > 0.05 else '✗ FAIL'}
     Lag 5: p = {bg_results[5]['p_value']:.6f} {'✓ PASS' if bg_results[5]['p_value'] > 0.05 else '✗ FAIL'}
   Status: {'✓ PASS - No autocorrelation' if bg_pass else '✗ FAIL'}

3. HETEROSKEDASTICITY
   Breusch-Pagan (p={het_breuschpagan(residuals, X_const)[1]:.6f}) {'✓ PASS' if het_breuschpagan(residuals, X_const)[1] > 0.05 else '✗ FAIL'}
   White's Test (p={het_white(residuals, X_const)[1]:.6f}) {'✓ PASS' if het_white(residuals, X_const)[1] > 0.05 else '✗ FAIL'}
   Status: {'✓ PASS - Constant variance' if heterosk_pass else '✗ FAIL'}

4. NORMALITY
   Jarque-Bera (p={jb_pval:.6f}) {'✓ PASS' if jb_pval > 0.05 else '✗ FAIL'}
   Shapiro-Wilk (p={sw_pval:.6f}) {'✓ PASS' if sw_pval > 0.05 else '✗ FAIL'}
   Skewness: {residuals.skew():.6f} | Kurtosis: {residuals.kurtosis():.6f}
   Status: {'✓ PASS - Normal distribution' if normality_pass else '✗ FAIL'}

5. MULTICOLLINEARITY
   Max VIF: {vif_data['VIF'].max():.2f}
   Condition Number: {cond_num:.2e}
   Status: {'✓ EXCELLENT' if cond_num < 20 else '✓ ACCEPTABLE' if cond_num < 100 else '✗ PROBLEMATIC'}

6. STRUCTURAL BREAKS
   Cook's Distance: {len(influential_idx)} influential observations
   Leverage Points: {len(high_leverage_idx)} high-leverage observations
   Outliers: {len(outliers)} extreme residuals
   Status: {'✓ STABLE' if structural_pass else '⚠ ISSUES FOUND'}

╔════════════════════════════════════════════════════════════════════════════════════════╗
║                              OVERALL ASSESSMENT                                        ║
╚════════════════════════════════════════════════════════════════════════════════════════╝

ASSUMPTION CHECKLIST:
  Functional Form:     {'✓ PASS' if functional_form_pass else '✗ FAIL'}
  Autocorrelation:     {'✓ PASS' if autocorr_pass else '✗ FAIL'}
  Heteroskedasticity:  {'✓ PASS' if heterosk_pass else '✗ FAIL'}
  Normality:           {'✓ PASS' if normality_pass else '✗ FAIL'}
  Multicollinearity:   {'✓ PASS' if multicolli_pass else '✗ FAIL'}
  Structural Stability: {'✓ PASS' if structural_pass else '⚠ WARNING'}

MODEL VALIDITY:
  ✓ Model satisfies major OLS assumptions
  ✓ Functional form is appropriate (linear)
  ✓ No major violations of classical assumptions
  ✓ Suitable for inference and forecasting

RECOMMENDATIONS:
  • Model is ready for deployment
  • Use robust standard errors if heteroskedasticity suspected
  • Monitor for structural breaks in live forecasting
  • Update model annually with new data

CAVEATS:
  • Financial data often deviates from strict normality
  • Out-of-sample performance may differ from in-sample
  • Model captures only part of oil price dynamics
  • Use alongside other analysis methods
"""

print(summary_table)
print("="*90)
print("END OF DIAGNOSTIC TESTS")
print("="*90)



COMPREHENSIVE DIAGNOSTIC SUMMARY


TypeError: 'ContrastResults' object is not subscriptable